# Tuning Hyperparameter - Image Captioning

Bagian optimasi tahap 3. Membandingkan training awal (**exp01**, Anggota 3) dengan 2 percobaan baru. Satu percobaan hanya mengubah **satu** pengaturan, supaya pengaruhnya jelas.

| Percobaan | Yang diubah | Alasan |
|---|---|---|
| exp01 (baseline) | lr 1e-3, dropout 0.3 | Hasil training Anggota 3 |
| exp02 | **learning rate 5e-4** | Di exp01 val loss baru turun lagi setelah LR diturunkan scheduler (epoch 27), jadi LR awal mungkin terlalu besar |
| exp03 | **dropout 0.5** | Di exp01 jarak train-val melebar setelah epoch ~7 (mulai overfitting), jadi regularisasi lebih kuat mungkin membantu |

Split data, seed (42), dan pengaturan lain sama dengan exp01, supaya perbandingannya adil.

**Cara menjalankan:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all**.

Checkpoint dan hasil disimpan di Google Drive. **Jika Colab terputus, cukup Run all lagi**: percobaan yang sudah selesai dilewati, dan yang belum selesai dilanjutkan dari epoch terakhir.

## 1. Setup: Drive, repo, dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/steezym/image-captioning.git'
BRANCH   = 'main'
REPO     = Path('/content/image-captioning')
DRIVE    = Path('/content/drive/MyDrive/ImageCaptioning')
ARCHIVE  = DRIVE / 'data' / 'archive.zip'
DATA     = Path('/content/data')

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'pull', '-q', 'origin', BRANCH], check=True)
os.chdir(REPO)

# Ekstrak dataset ke disk lokal Colab (jauh lebih cepat daripada membaca dari Drive)
if not (DATA / 'Images').is_dir():
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as z:
        z.extractall(DATA)
print('Gambar  :', len(os.listdir(DATA / 'Images')))   # harus 8091
print('Captions:', (DATA / 'captions.txt').is_file())

import torch
print('GPU     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'TIDAK ADA, ganti runtime ke T4 GPU')

## 2. Daftar percobaan

In [ ]:
# Pengaturan umum = default train.py (sama dengan exp01)
UMUM = ['--epochs', '30', '--batch-size', '64', '--seed', '42', '--num-workers', '2',
        '--device', 'cuda', '--progress-every', '100']

PERCOBAAN = {
    'exp02_lr5e-4':     ['--learning-rate', '5e-4'],
    'exp03_dropout0.5': ['--dropout', '0.5'],
}

def folder(nama):
    return DRIVE / 'checkpoints' / nama, DRIVE / 'results' / nama

## 3. Training (bisa dilanjutkan jika terputus)

Perkiraan waktu: puluhan menit sampai beberapa jam per percobaan, tergantung kecepatan GPU dan pembacaan data. Early stopping aktif (berhenti jika val loss tidak membaik selama 5 epoch).

In [ ]:
import sys

def jalankan(perintah):
    print('$ python -m', ' '.join(perintah))
    subprocess.run([sys.executable, '-m', *perintah], check=True)

for nama, ubah in PERCOBAAN.items():
    ck, res = folder(nama)
    ck.mkdir(parents=True, exist_ok=True)
    res.mkdir(parents=True, exist_ok=True)

    if (res / 'training_history.json').is_file():
        print(f'== {nama}: training sudah selesai, dilewati')
        continue

    perintah = ['src.train',
                '--images-dir', str(DATA / 'Images'), '--captions-file', str(DATA / 'captions.txt'),
                '--checkpoint-dir', str(ck), '--results-dir', str(res), *UMUM, *ubah]
    if (ck / 'last_checkpoint.pt').is_file():
        print(f'== {nama}: melanjutkan dari checkpoint terakhir')
        perintah += ['--resume', str(ck / 'last_checkpoint.pt')]
    else:
        print(f'== {nama}: mulai training')
    jalankan(perintah)

## 4. Evaluasi tiap percobaan pada test set (BLEU, METEOR, CIDEr)

In [ ]:
for nama in PERCOBAAN:
    ck, res = folder(nama)
    if (res / 'evaluation.json').is_file():
        print(f'== {nama}: evaluasi sudah ada, dilewati')
        continue
    jalankan(['src.evaluate',
              '--images-dir', str(DATA / 'Images'), '--captions-file', str(DATA / 'captions.txt'),
              '--checkpoint', str(ck / 'best_model.pt'), '--output', str(res / 'evaluation.json'),
              '--device', 'cuda', '--num-workers', '2', '--samples', '10'])

## 5. Perbandingan hasil

exp01 diambil dari hasil Anggota 3 di repo (`results/`).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

sumber = {'exp01_baseline': REPO / 'results'}
sumber.update({nama: folder(nama)[1] for nama in PERCOBAAN})

baris, riwayat = [], {}
for nama, res in sumber.items():
    hist = json.loads((res / 'training_history.json').read_text())
    ev = json.loads((res / 'evaluation.json').read_text())
    terbaik = min(hist, key=lambda r: r['val_loss'])
    riwayat[nama] = hist
    baris.append({
        'percobaan': nama,
        'epoch dijalankan': len(hist),
        'epoch terbaik': terbaik['epoch'],
        'val loss terbaik': round(terbaik['val_loss'], 4),
        'test loss': round(ev['test_loss'], 4),
        **{k.upper(): round(v, 4) for k, v in ev['metrics'].items()},
    })

tabel = pd.DataFrame(baris).set_index('percobaan')
display(tabel)
tabel.to_csv(DRIVE / 'results' / 'tuning_perbandingan.csv')

fig, ax = plt.subplots(figsize=(9, 5))
for nama, hist in riwayat.items():
    ax.plot([r['epoch'] for r in hist], [r['val_loss'] for r in hist], marker='o', ms=3, label=nama)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation loss')
ax.set_title('Perbandingan validation loss')
ax.grid(alpha=.3)
ax.legend()
fig.savefig(DRIVE / 'results' / 'tuning_val_loss.png', dpi=150, bbox_inches='tight')
plt.show()

terbaik = tabel['val loss terbaik'].idxmin()
print('Percobaan dengan val loss terendah:', terbaik)
print('Disimpan di:', DRIVE / 'results')

## 6. Contoh caption dari percobaan terbaik

In [ ]:
ev = json.loads((sumber[terbaik] / 'evaluation.json').read_text())
for s in ev['samples'][:10]:
    print(s['image'])
    print('  model:', s['prediction'])
    print('  asli :', s['references'][0])
    print()